In [2]:
!pip install spacy

  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/15.3 MB ? eta -:--:--
    --------------------------------------- 0.3/15.3 MB 9.9 MB/s eta 0:00:02
   - -------------------------------------- 0.7/15.3 MB 8.8 MB/s eta 0:00:02
   -- ------------------------------------- 0.9/15.3 MB 7.0 MB/s eta 0:00:03
   -- ------------------------------------- 1.0/15.3 MB 6.0 MB/s eta 0:00:03
   --- ------------------------------------ 1.2/15.3 MB 5.5 MB/s eta 0:00:03
   --- ------------------------------------ 1.4/15.3 MB 5.2 MB/s eta 0:00:03
   ---- ----------------------------------- 1.6/15.3 MB 5.0 MB/s eta 0:00:03
   ---- ----------------------------------- 1.8/15.3 MB 4.9 MB/s eta 0:00:03
   ---- ----------------------------------- 1.9/15.3 MB 4.9 MB/s eta 0:00:03
   ----- ----------------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
anaconda-cli-base 0.5.4 requires click<8.2, but you have click 8.3.1 which is incompatible.
chromadb 1.5.2 requires tenacity>=8.2.3, but you have tenacity 8.2.2 which is incompatible.
gtts 2.5.2 requires click<8.2,>=7.1, but you have click 8.3.1 which is incompatible.


In [3]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------------------------------------- 0.0/12.8 MB 262.6 kB/s eta 0:00:49
     --------------------------------------- 0.1/12.8 MB 409.6 kB/s eta 0:00:32
     - -------------------------------------- 0.4/12.8 MB 2.2 MB/s eta 0:00:06
     -- ------------------------------------- 0.8/12.8 MB 3.6 MB/s eta 0:00:04
     --- ------------------------------------ 1.0/12.8 MB 3.9 MB/s eta 0:00:04
     --- ------------------------------------ 1.2/12.8 MB 3.7 MB/s eta 0:00:04
     --- ------------------------------------ 1.3/12.8 MB 3.8 MB/s eta 0:00:04
     --- ------------------------------------ 1.3/12.8 MB 3.8 MB/s eta 0:00:04
     ---- ----------------------------------- 1.3/12.8 MB 2.9 MB/s eta 0:00:04
     ----- ---------------------------------- 1.8/12.8 MB 3.6 MB/s eta 0:00:04
     ------ --------------------------------- 2.0/12.8 MB 3.8 MB

In [5]:
!pip install pdfminer.six

   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB 217.9 kB/s eta 0:00:31
   ---------------------------------------- 0.1/6.6 MB 363.1 kB/s eta 0:00:18
   - -------------------------------------- 0.3/6.6 MB 1.6 MB/s eta 0:00:04
   --- ------------------------------------ 0.6/6.6 MB 2.7 MB/s eta 0:00:03
   ------ --------------------------------- 1.0/6.6 MB 3.4 MB/s eta 0:00:02
   ------- -------------------------------- 1.2/6.6 MB 3.5 MB/s eta 0:00:02
   ------- -------------------------------- 1.3/6.6 MB 3.4 MB/s eta 0:00:02
   ------- -------------------------------- 1.3/6.

In [7]:
import pandas as pd
import numpy as np
import spacy
import re
import os

from pdfminer.high_level import extract_text
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
jobs = pd.read_csv(r"C:\Users\DELL\Documents\AI\AI_Resume_Screener\dataset\job_descriptions_dataset.csv")
jobs.head()

,job_title,skills,job_description
0,Data Scientist,"Python, Machine Learning, SQL, Statistics, Pan...","Analyze large datasets, build predictive model..."
1,Machine Learning Engineer,"Python, Machine Learning, TensorFlow, PyTorch,...","Design, build, and deploy scalable machine lea..."
2,Data Analyst,"SQL, Excel, Python, Data Visualization, Tablea...","Analyze business data, build dashboards, and g..."
3,AI Engineer,"Python, Deep Learning, TensorFlow, PyTorch, NL...",Develop artificial intelligence systems using ...
4,Python Developer,"Python, Django, Flask, APIs, SQL, Git",Develop backend systems and APIs using Python ...


In [12]:
#Select Job Role
job_role = "Data Scientist"
job_description = jobs[jobs['job_title']==job_role]['skills'].iloc[0]
print(job_description)

Python, Machine Learning, SQL, Statistics, Pandas, Scikit-learn, Data Visualization, NLP


In [13]:
#Extract Text From Resume
def extract_resume_text(file_path):
    text = extract_text(file_path)
    return text

In [14]:
resume_text = extract_resume_text(r"C:\Users\DELL\Documents\AI\AI_Resume_Screener\dataset\resumes\resume1.pdf")
print(resume_text[:500])

Amit Sharma

Role: Data Scientist

Professional Summary

Worked on predictive modeling and data analysis projects using Python and machine learning.

Skills

Python

Machine Learning

SQL

Pandas

Scikit-learn

Data Visualization

Statistics

NLP




In [15]:
#Clean Resume Text
def clean_text(text):

    text = re.sub(r'\n',' ',text)

    text = re.sub(r'[^A-Za-z ]','',text)

    text = text.lower()

    return text

In [16]:
#Load NLP Model
nlp = spacy.load("en_core_web_sm")

In [17]:
#Skill Extraction
skills_list = [
    "python","machine learning","sql","deep learning",
    "data analysis","nlp","tensorflow","pandas","java"
]

def extract_skills(text):

    doc = nlp(text)

    found_skills = []

    for token in doc:

        if token.text.lower() in skills_list:

            found_skills.append(token.text)

    return list(set(found_skills))

In [18]:
#Similarity Model
def calculate_similarity(resume, job_description):

    vectorizer = TfidfVectorizer()

    tfidf_matrix = vectorizer.fit_transform([resume, job_description])

    score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])

    return score[0][0]

In [ ]:
#Rank Multiple Resumes
resume_folder = r"C:\Users\DELL\Documents\AI\AI_Resume_Screener\dataset\resumes"

results = []

for file in os.listdir(resume_folder):
    file_path = os.path.join(resume_folder,file)
    resume_text = extract_resume_text(file_path)
    cleaned = clean_text(resume_text)
    score = calculate_similarity(cleaned, job_description)
    results.append((file,score))

In [21]:
#Sort Candidates
ranked = sorted(results,key=lambda x:x[1],reverse=True)
for name,score in ranked:
    print(name,score)

resume1.pdf 0.4880767302742728
resume3.pdf 0.22794900336940474
resume4.pdf 0.19800517126347725
resume5.pdf 0.13387083899186267
resume2.pdf 0.11671770522446966


In [22]:
#Show Results Table
df = pd.DataFrame(ranked,columns=["Candidate Resume","Match Score"])
df

,Candidate Resume,Match Score
0,resume1.pdf,0.488077
1,resume3.pdf,0.227949
2,resume4.pdf,0.198005
3,resume5.pdf,0.133871
4,resume2.pdf,0.116718
